# LeetCode 2891 — Method Chaining bằng Python (pandas)

Link đề: https://leetcode.com/problems/method-chaining/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ bảng `animals` (`name`, `species`, `age`, `weight`), **lọc các con vật nặng hơn 100 kg**, **sắp xếp theo `weight` giảm dần**, rồi **chỉ trả về cột `name`**, và làm tất cả bằng **một chuỗi lệnh nối nhau (method chaining)**.
2. Hiểu **method chaining**: mỗi phương thức của DataFrame **trả về một DataFrame mới**, nên gọi nối tiếp `df.lọc().sắp_xếp().chọn_cột()` được.
3. Đi từ cách **vòng lặp thuần Python (lọc, sắp xếp, dựng bảng)** → **mặt nạ + `sort_values` + chọn cột** → **`query`** → **`loc` với `lambda`** → **sắp xếp trước rồi lọc, dùng `pipe`/`filter`**.
4. Biết **thứ tự các bước quan trọng** (phải giữ cột `weight` cho tới khi lọc/sắp xếp xong, rồi mới chọn `name`) và **xử lý hàng có `weight` bằng nhau** (sắp xếp ổn định).
5. Tránh các lỗi kinh điển: `>=` thay vì `>`, quên `ascending=False`, chọn cột quá sớm, trả về Series thay vì DataFrame, dùng `inplace=True` làm đứt chuỗi.

## 2. Đề bài & ví dụ

Có một bảng `animals` gồm các cột `name`, `species`, `age`, `weight` (cân nặng, đơn vị kg).

Hãy viết hàm trả về **tên của những con vật nặng hơn 100 kg**, **sắp xếp theo cân nặng giảm dần**. Kết quả là một DataFrame chỉ có **một cột `name`**. Hãy giải bằng **method chaining**.

**Ví dụ**

- Input: DataFrame `animals`

| name | species | age | weight |
|---|---|---|---|
| Tatiana | Snake | 98 | 464 |
| Khaled | Giraffe | 50 | 41 |
| Alex | Leopard | 6 | 328 |
| Jonathan | Monkey | 45 | 463 |
| Stefan | Bear | 100 | 50 |
| Tommy | Panda | 26 | 349 |

- Output:

| name |
|---|
| Tatiana |
| Jonathan |
| Tommy |
| Alex |

- Giải thích: bốn con vật nặng hơn 100 kg là Tatiana (464), Jonathan (463), Tommy (349), Alex (328); sắp xếp giảm dần theo cân nặng. Khaled (41) và Stefan (50) bị loại.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc. Việc kết quả giữ nhãn hàng gốc hay đánh lại `0..n-1` cũng nên đối chiếu với đề gốc; ở notebook này ta giữ nhãn gốc, đúng mặc định của các phép lọc/sắp xếp.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

animals = pd.DataFrame({
    "name": ["Tatiana", "Khaled", "Alex", "Jonathan", "Stefan", "Tommy"],
    "species": ["Snake", "Giraffe", "Leopard", "Monkey", "Bear", "Panda"],
    "age": [98, 50, 6, 45, 100, 26],
    "weight": [464, 41, 328, 463, 50, 349],
})
print(animals)
print()
print(animals.dtypes)
print("pandas", pd.__version__)

       name  species  age  weight
0   Tatiana    Snake   98     464
1    Khaled  Giraffe   50      41
2      Alex  Leopard    6     328
3  Jonathan   Monkey   45     463
4    Stefan     Bear  100      50
5     Tommy    Panda   26     349

name         str
species      str
age        int64
weight     int64
dtype: object
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Method chaining là gì?

Hầu hết phương thức của DataFrame **không sửa bảng cũ** mà **trả về một DataFrame mới**. Vì vậy ta có thể **nối** lệnh: kết quả của lệnh trước là đối tượng để gọi lệnh sau.

```python
df.lọc(...).sắp_xếp(...).chọn_cột(...)
```

Mỗi `.` là một bước biến đổi. Đọc từ trái sang phải như một dây chuyền. Khi chuỗi dài, ta đặt trong **dấu ngoặc đơn `( ... )`** và xuống dòng, mỗi dòng một bước.

In [2]:
step1 = animals[animals["weight"] > 100]              # một bước
step2 = step1.sort_values("weight", ascending=False)   # bước tiếp theo
step3 = step2[["name"]]                                # bước cuối
print(step3)

chained = (
    animals[animals["weight"] > 100]
    .sort_values("weight", ascending=False)
    [["name"]]
)
print(chained.equals(step3))                           # True: nối lại thành một chuỗi, kết quả y hệt

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex
True


### 3.2. Lọc bằng mặt nạ boolean

Một biểu thức so sánh trên cột (`animals["weight"] > 100`) cho **Series `True/False`** (mặt nạ). Đặt vào `animals[mặt_nạ]` thì pandas **giữ các hàng `True`**. Dấu so sánh quan trọng: đề nói **"nặng hơn 100"** nên là `>` (lớn hơn **hẳn**), không phải `>=`.

In [3]:
mask = animals["weight"] > 100
print(mask.tolist())                  # [True, False, True, True, False, True]
print(animals[mask]["name"].tolist())
print((animals["weight"] >= 100).sum(), "vs", (animals["weight"] > 100).sum())   # 100 chính xác sẽ khác nhau

[True, False, True, True, False, True]
['Tatiana', 'Alex', 'Jonathan', 'Tommy']
4 vs 4


### 3.3. `sort_values(by, ascending=...)`

`df.sort_values("weight")` sắp xếp các **hàng** theo cột `weight` **tăng dần** (mặc định). `ascending=False` sắp xếp **giảm dần**. Hàm trả về bảng mới; các hàng **giữ nhãn gốc** (nhãn bị xáo trộn theo thứ tự mới).

In [4]:
print(animals.sort_values("weight")["name"].tolist())                     # tăng dần
print(animals.sort_values("weight", ascending=False)["name"].tolist())    # giảm dần
print(animals.sort_values("weight", ascending=False).index.tolist())      # nhãn gốc bị xáo trộn

['Khaled', 'Stefan', 'Alex', 'Tommy', 'Jonathan', 'Tatiana']
['Tatiana', 'Jonathan', 'Tommy', 'Alex', 'Stefan', 'Khaled']
[0, 3, 5, 2, 4, 1]


### 3.4. Chọn cột: `[["name"]]` (DataFrame) và `["name"]` (Series)

- `df["name"]` (một cặp ngoặc vuông, một tên) trả về **Series** (một cột).
- `df[["name"]]` (**hai** cặp ngoặc, một danh sách) trả về **DataFrame** có một cột.

Đề yêu cầu kết quả là DataFrame một cột, nên dùng `[["name"]]`.

In [5]:
print(type(animals["name"]).__name__)        # Series
print(type(animals[["name"]]).__name__)      # DataFrame
print(animals[["name"]].shape)               # (6, 1)

Series
DataFrame
(6, 1)


### 3.5. `DataFrame.query("điều kiện")`

`df.query("weight > 100")` lọc bằng **chuỗi biểu thức**, viết ngắn hơn mặt nạ và không phải gõ lại `animals[...]`; rất hợp với method chaining vì không cần tên biến. Tên cột viết thẳng trong chuỗi; hằng số Python dùng ký tự `@` (ví dụ `query("weight > @limit")`).

In [6]:
print(animals.query("weight > 100")["name"].tolist())
limit = 300
print(animals.query("weight > @limit")["name"].tolist())      # @limit = biến Python bên ngoài

['Tatiana', 'Alex', 'Jonathan', 'Tommy']
['Tatiana', 'Alex', 'Jonathan', 'Tommy']


### 3.6. `loc` với `lambda` và `pipe`

- `df.loc[lambda d: d["weight"] > 100]`: `lambda` nhận **bảng hiện tại `d`** (kết quả của bước trước trong chuỗi), nên không cần biết tên biến.
- `df.pipe(hàm)`: đưa **cả bảng** vào một hàm tự viết rồi nhận kết quả; giúp **chèn hàm tùy biến** vào giữa chuỗi.
- `df.filter(items=["name"])`: chọn cột theo **danh sách tên**, trả DataFrame.

In [7]:
print(animals.loc[lambda d: d["weight"] > 100].shape)           # (4, 4)

def heavy_only(d):
    return d[d["weight"] > 100]

print(animals.pipe(heavy_only).shape)                           # (4, 4)
print(animals.filter(items=["name"]).columns.tolist())          # ['name']

(4, 4)
(4, 4)
['name']


### 3.7. Hàng có `weight` bằng nhau: sắp xếp ổn định

Nếu hai con vật có **cùng cân nặng**, thứ tự giữa chúng chưa xác định nếu thuật toán sắp xếp không **ổn định (stable)**. `sort_values(..., kind="stable")` **giữ thứ tự ban đầu** của các hàng bằng nhau. Đề không nói rõ cách xử lý, nên ta dùng `kind="stable"` để kết quả **xác định** và khớp cách duyệt trong vòng lặp.

In [8]:
tie = pd.DataFrame({"name": ["a", "b", "c", "d"], "weight": [200, 300, 200, 300]})
print(tie.sort_values("weight", ascending=False, kind="stable")["name"].tolist())   # b, d, a, c: giữ thứ tự cũ trong mỗi nhóm bằng nhau

['b', 'd', 'a', 'c']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy làm như một người quản lý sở thú cầm danh sách: **gạch tên những con nhẹ**, rồi **xếp những con còn lại từ nặng nhất đến nhẹ nhất**, rồi **chép riêng cột tên**.

| bước | nhãn | name | weight | nặng hơn 100? | giữ? |
|---|---|---|---|---|---|
| 1 | 0 | Tatiana | 464 | có | giữ |
| 2 | 1 | Khaled | 41 | không | bỏ |
| 3 | 2 | Alex | 328 | có | giữ |
| 4 | 3 | Jonathan | 463 | có | giữ |
| 5 | 4 | Stefan | 50 | không | bỏ |
| 6 | 5 | Tommy | 349 | có | giữ |

Giữ lại: Tatiana (464), Alex (328), Jonathan (463), Tommy (349). **Xếp giảm dần theo weight:** 464 Tatiana, 463 Jonathan, 349 Tommy, 328 Alex. **Chỉ lấy tên:** Tatiana, Jonathan, Tommy, Alex.

Thuật toán gồm ba việc theo thứ tự: **lọc -> sắp xếp -> chọn cột**. Method chaining chỉ là cách viết ba việc đó thành một dây chuyền.

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

**Ý tưởng.** Duyệt từng hàng; hàng nào có `weight > 100` thì ghi lại cặp `(weight, nhãn)`. Sắp xếp các cặp theo `weight` giảm dần (hàm `sorted` của Python ổn định). Cuối cùng lấy các hàng theo danh sách nhãn đã sắp và chỉ giữ cột `name`.

In [9]:
def heavy_loop(animals: pd.DataFrame) -> pd.DataFrame:
    heavy = []                                              # các cặp (weight, nhãn)
    for label, weight in zip(animals.index, animals["weight"]):
        if weight > 100:                                    # nặng hơn 100 kg (lớn hơn hẳn)
            heavy.append((weight, label))
    heavy.sort(key=lambda pair: -pair[0])                   # giảm dần theo weight; sort của Python ổn định
    ordered_labels = [label for _, label in heavy]
    return animals.loc[ordered_labels, ["name"]]            # lấy hàng theo nhãn, chỉ cột name


print(heavy_loop(animals))

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex


In [10]:
res = heavy_loop(animals)
print("tên   :", res["name"].tolist())              # ['Tatiana', 'Jonathan', 'Tommy', 'Alex']
print("nhãn  :", res.index.tolist())                # [0, 3, 5, 2]
print("kiểu  :", type(res).__name__, res.shape)

tên   : ['Tatiana', 'Jonathan', 'Tommy', 'Alex']
nhãn  : [0, 3, 5, 2]
kiểu  : DataFrame (4, 1)


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng, `h` là số con vật nặng hơn 100. Lọc `O(m)`, sắp xếp `O(h log h)`, lấy hàng `O(h)`: **Time `O(m + h log h)`**; danh sách `heavy` và bảng kết quả: **Space `O(h)`**.

**Nút thắt.** Vòng lặp Python chậm khi `m` lớn và phải tự lo nhiều chi tiết. pandas có sẵn `sort_values`, mặt nạ, chọn cột; gộp lại bằng method chaining.

## 6. Ý tưởng thuật toán / bước đột phá

Ba việc lọc -> sắp xếp -> chọn cột đều có hàm sẵn trong pandas và **mỗi hàm trả về DataFrame mới**, nên nối thành chuỗi:

```python
(animals[animals["weight"] > 100]
        .sort_values("weight", ascending=False)
        [["name"]])
```

- `animals[animals["weight"] > 100]`: lọc theo mặt nạ;
- `.sort_values("weight", ascending=False)`: sắp xếp giảm dần;
- `[["name"]]`: chọn cột `name`, **cuối cùng** (vì cần cột `weight` cho đến khi sắp xếp xong).

Các cách khác cho cùng kết quả: **`query`** (Cách 3), **`loc` + `lambda`** (Cách 4), **sắp xếp trước rồi lọc, dùng `pipe` / `filter`** (Cách 5).

## 7. Các cách tiếp theo

### Cách 2 — Mặt nạ + `sort_values` + chọn cột (cách chuẩn)

**Giải thích.** Một chuỗi ba bước, không sửa bảng đầu vào. Ta thêm `kind="stable"` để các con vật cùng cân nặng giữ thứ tự ban đầu.

In [11]:
def heavy_mask(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals[animals["weight"] > 100]
        .sort_values("weight", ascending=False, kind="stable")
        [["name"]]
    )


print(heavy_mask(animals))

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals[animals["weight"] > 100]
        .sort_values("weight", ascending=False)
        [["name"]]
    )

**Độ phức tạp Cách 2:** **Time `O(m + h log h)`** (lọc `m` hàng, sắp xếp `h` hàng còn lại), **Space `O(m + h)`** (mặt nạ `m` phần tử bool và bảng đã lọc).

### Cách 3 — `query`

**Giải thích.** `query("weight > 100")` thay mặt nạ; chuỗi không phải nhắc lại tên biến `animals`, nên đọc rất gọn. Phần còn lại giống Cách 2.

In [13]:
def heavy_query(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals.query("weight > 100")
        .sort_values("weight", ascending=False, kind="stable")
        [["name"]]
    )


print(heavy_query(animals))

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame:
    return animals.query("weight > 100").sort_values("weight", ascending=False)[["name"]]

**Độ phức tạp Cách 3:** **Time `O(m + h log h)`**, **Space `O(m + h)`**.

### Cách 4 — `loc[lambda d: ...]` và chọn cột ngay trong `loc`

**Giải thích.** `loc[hàng, cột]` làm **hai việc một lúc**: lọc hàng bằng `lambda` và chọn cột `["name"]`. Sau đó chỉ còn `sort_values`... nhưng **cột `weight` đã bị bỏ**, nên ta phải sắp xếp **trước** khi chọn cột. Vì vậy thứ tự đúng là: `loc` lọc (giữ mọi cột) -> `sort_values` -> `loc[:, ["name"]]`.

In [15]:
def heavy_loc(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals
        .loc[lambda d: d["weight"] > 100]
        .sort_values("weight", ascending=False, kind="stable")
        .loc[:, ["name"]]
    )


print(heavy_loc(animals))

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex


In [16]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals
        .loc[lambda d: d["weight"] > 100]
        .sort_values("weight", ascending=False)
        .loc[:, ["name"]]
    )

**Độ phức tạp Cách 4:** **Time `O(m + h log h)`**, **Space `O(m + h)`**.

### Cách 5 — Sắp xếp trước rồi lọc; dùng `pipe` và `filter`

**Giải thích.** Thứ tự **sắp xếp rồi lọc** cho cùng kết quả (lọc không đổi thứ tự tương đối của các hàng còn lại). Ta chèn một hàm tự viết bằng `pipe`, và chọn cột bằng `filter(items=[...])`. Cách này kém hiệu quả hơn (sắp xếp cả `m` hàng thay vì `h`) nhưng cho thấy **hai phép biến đổi độc lập thì đổi chỗ được**.

In [17]:
def heavy_pipe(animals: pd.DataFrame) -> pd.DataFrame:
    def only_heavy(d: pd.DataFrame) -> pd.DataFrame:
        return d[d["weight"] > 100]

    return (
        animals
        .sort_values("weight", ascending=False, kind="stable")     # sắp xếp cả bảng trước
        .pipe(only_heavy)                                           # rồi lọc bằng hàm tự viết
        .filter(items=["name"])                                     # cuối cùng chọn cột
    )


print(heavy_pipe(animals))

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex


**Độ phức tạp Cách 5:** **Time `O(m log m)`** (sắp xếp cả bảng), **Space `O(m)`**.

> Cách 2 và Cách 3 là cách nên nộp: ngắn và rõ. Nhớ ba điều: `>` (không phải `>=`), `ascending=False`, chọn cột `[["name"]]` **sau cùng**.

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết; (b) từng mắt xích của chuỗi: lọc -> sắp xếp -> chọn cột.

In [18]:
def trace_loop(animals):
    heavy = []
    for step, (label, name, weight) in enumerate(zip(animals.index, animals["name"], animals["weight"]), start=1):
        keep = weight > 100
        print(f"bước {step}: nhãn={label}, name={name:<9} weight={weight:>3} -> {'giữ' if keep else 'bỏ'}")
        if keep:
            heavy.append((weight, label))
        print(f"          heavy = {heavy}")
    heavy.sort(key=lambda pair: -pair[0])
    print("\nsắp xếp giảm dần theo weight:", heavy)
    labels = [label for _, label in heavy]
    out = animals.loc[labels, ["name"]]
    print("kết quả (nhãn, name):", list(zip(out.index, out["name"])))
    return out


_ = trace_loop(animals)

bước 1: nhãn=0, name=Tatiana   weight=464 -> giữ
          heavy = [(464, 0)]
bước 2: nhãn=1, name=Khaled    weight= 41 -> bỏ
          heavy = [(464, 0)]
bước 3: nhãn=2, name=Alex      weight=328 -> giữ
          heavy = [(464, 0), (328, 2)]
bước 4: nhãn=3, name=Jonathan  weight=463 -> giữ
          heavy = [(464, 0), (328, 2), (463, 3)]
bước 5: nhãn=4, name=Stefan    weight= 50 -> bỏ
          heavy = [(464, 0), (328, 2), (463, 3)]
bước 6: nhãn=5, name=Tommy     weight=349 -> giữ
          heavy = [(464, 0), (328, 2), (463, 3), (349, 5)]

sắp xếp giảm dần theo weight: [(464, 0), (463, 3), (349, 5), (328, 2)]
kết quả (nhãn, name): [(0, 'Tatiana'), (3, 'Jonathan'), (5, 'Tommy'), (2, 'Alex')]


In [19]:
def trace_chain(animals):
    print("BƯỚC 0 - bảng gốc", animals.shape)
    print(animals)

    mask = animals["weight"] > 100
    print("\nBƯỚC 1a - mặt nạ weight > 100:", mask.tolist())
    filtered = animals[mask]
    print("BƯỚC 1b - sau khi lọc:", filtered.shape, "| nhãn:", filtered.index.tolist())
    print(filtered)

    ordered = filtered.sort_values("weight", ascending=False, kind="stable")
    print("\nBƯỚC 2 - sort_values('weight', ascending=False):")
    print(ordered)
    print("nhãn:", ordered.index.tolist(), "(xáo trộn theo thứ tự mới)")

    result = ordered[["name"]]
    print("\nBƯỚC 3 - chọn cột [['name']]:", result.shape)
    print(result)

    same = result.equals(animals.query("weight > 100").sort_values("weight", ascending=False, kind="stable")[["name"]])
    print("\ngiống cách query?", same)
    return result


_ = trace_chain(animals)

BƯỚC 0 - bảng gốc (6, 4)
       name  species  age  weight
0   Tatiana    Snake   98     464
1    Khaled  Giraffe   50      41
2      Alex  Leopard    6     328
3  Jonathan   Monkey   45     463
4    Stefan     Bear  100      50
5     Tommy    Panda   26     349

BƯỚC 1a - mặt nạ weight > 100: [True, False, True, True, False, True]
BƯỚC 1b - sau khi lọc: (4, 4) | nhãn: [0, 2, 3, 5]
       name  species  age  weight
0   Tatiana    Snake   98     464
2      Alex  Leopard    6     328
3  Jonathan   Monkey   45     463
5     Tommy    Panda   26     349

BƯỚC 2 - sort_values('weight', ascending=False):
       name  species  age  weight
0   Tatiana    Snake   98     464
3  Jonathan   Monkey   45     463
5     Tommy    Panda   26     349
2      Alex  Leopard    6     328
nhãn: [0, 3, 5, 2] (xáo trộn theo thứ tự mới)

BƯỚC 3 - chọn cột [['name']]: (4, 1)
       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex

giống cách query? True


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | `animals` | `(6, 4)` |
| 1 | `animals[weight > 100]` | `(4, 4)`, nhãn `[0, 2, 3, 5]` |
| 2 | `.sort_values("weight", ascending=False)` | `(4, 4)`, nhãn `[0, 3, 5, 2]` |
| 3 | `[["name"]]` | `(4, 1)`: Tatiana, Jonathan, Tommy, Alex |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [20]:
import pandas as pd

def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals[animals["weight"] > 100]
        .sort_values("weight", ascending=False)
        [["name"]]
    )


print(findHeavyAnimals(animals))

       name
0   Tatiana
3  Jonathan
5     Tommy
2      Alex


## 10. Vì sao thuật toán đúng

- **Lọc đúng:** mặt nạ `weight > 100` giữ đúng những hàng nặng hơn hẳn 100 kg; con vật đúng 100 kg bị loại như đề yêu cầu ("nặng hơn").
- **Sắp xếp đúng:** `sort_values("weight", ascending=False)` xếp các hàng còn lại theo `weight` từ lớn đến nhỏ; với `kind="stable"` các hàng bằng nhau giữ thứ tự ban đầu nên kết quả xác định.
- **Chọn cột sau cùng:** `weight` cần có mặt tới khi sắp xếp xong; chọn `name` ở bước cuối để chỉ trả về đúng cột đề yêu cầu.
- **Kiểu trả về:** `[["name"]]` (danh sách bên trong) cho DataFrame một cột, đúng chữ ký.
- **Không sửa đầu vào:** mọi bước trả bảng mới nên `animals` nguyên vẹn.
- **Tính chất kiểm được:** mọi giá trị `weight` của hàng kết quả đều `> 100`; dãy `weight` tương ứng giảm dần (không tăng); và tập tên kết quả đúng bằng tập tên có `weight > 100` của bảng gốc.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `h` là số hàng thỏa `weight > 100`.

- **Time:** lọc `O(m)` + sắp xếp `O(h log h)` + chọn cột `O(h)` = **`O(m + h log h)`**; trường hợp xấu nhất `h = m` là `O(m log m)`.
- **Space:** mặt nạ `O(m)`, bảng đã lọc và sắp xếp `O(h)` (mỗi bước chuỗi tạo một bảng trung gian mới).
- Sắp xếp theo so sánh không thể nhanh hơn `O(h log h)` trong trường hợp tổng quát; phần lọc không thể nhanh hơn `O(m)`.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`. Các hàm không sửa bảng đầu vào. Với test có `weight` bằng nhau, ta chỉ dùng các cách có `kind="stable"` (Cách 1 đến 5); bản nộp của LeetCode (`submit`) không bắt buộc ổn định nên chỉ kiểm khi `weight` không trùng.

In [21]:
impls = {
    "loop": heavy_loop,
    "mask": heavy_mask,
    "query": heavy_query,
    "loc": heavy_loc,
    "pipe": heavy_pipe,
}

expected_example = pd.DataFrame({"name": ["Tatiana", "Jonathan", "Tommy", "Alex"]}, index=[0, 3, 5, 2])

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(animals), expected_example)
pd.testing.assert_frame_equal(findHeavyAnimals(animals), expected_example)
print("PASS ví dụ của đề (mọi cách và bản nộp)")

PASS ví dụ của đề (mọi cách và bản nộp)


### 12.1. Các trường hợp biên

In [22]:
def make(rows):
    """rows: danh sách (name, weight)."""
    return pd.DataFrame({
        "name": pd.Series([r[0] for r in rows], dtype="str"),
        "species": pd.Series(["x"] * len(rows), dtype="str"),
        "age": pd.Series([1] * len(rows), dtype="int64"),
        "weight": pd.Series([r[1] for r in rows], dtype="int64"),
    })


def check_all(df, expected_names, label, with_submit=True):
    for key, fn in impls.items():
        got = fn(df)["name"].tolist()
        assert got == expected_names, (label, key, got, expected_names)
    if with_submit:
        assert findHeavyAnimals(df)["name"].tolist() == expected_names, (label, "submit")
    print("PASS", label, "->", expected_names)


check_all(make([("a", 100)]), [], "đúng 100 kg bị loại (phải lớn hơn hẳn)")
check_all(make([("a", 101)]), ["a"], "101 kg được giữ")
check_all(make([("a", 5), ("b", 7)]), [], "không con nào nặng hơn 100")
check_all(make([("a", 150), ("b", 120), ("c", 300)]), ["c", "a", "b"], "toàn nặng, sắp giảm dần")
check_all(make([("a", 300), ("b", 200), ("c", 100), ("d", 99)]), ["a", "b"], "biên 100 và 99")
check_all(make([("a", 200), ("b", 300), ("c", 200), ("d", 300)]), ["b", "d", "a", "c"],
          "weight trùng nhau: giữ thứ tự ban đầu (ổn định)", with_submit=False)

PASS đúng 100 kg bị loại (phải lớn hơn hẳn) -> []
PASS 101 kg được giữ -> ['a']
PASS không con nào nặng hơn 100 -> []
PASS toàn nặng, sắp giảm dần -> ['c', 'a', 'b']
PASS biên 100 và 99 -> ['a', 'b']
PASS weight trùng nhau: giữ thứ tự ban đầu (ổn định) -> ['b', 'd', 'a', 'c']


### 12.2. Bảng rỗng, kiểu trả về, đầu vào, và phản ví dụ

In [23]:
# Bảng rỗng: kết quả rỗng nhưng vẫn là DataFrame có cột name
empty = make([])
for key, fn in impls.items():
    out = fn(empty)
    assert isinstance(out, pd.DataFrame) and out.columns.tolist() == ["name"] and len(out) == 0, key
print("PASS bảng rỗng -> DataFrame rỗng có cột name")

# Kiểu trả về là DataFrame một cột (không phải Series)
out = findHeavyAnimals(animals)
assert isinstance(out, pd.DataFrame) and out.columns.tolist() == ["name"]
print("PASS kiểu trả về là DataFrame, cột:", out.columns.tolist())

# Tính chất: mọi weight kết quả > 100 và giảm dần
heavy_rows = animals.loc[out.index]
assert (heavy_rows["weight"] > 100).all()
assert heavy_rows["weight"].is_monotonic_decreasing
assert set(out["name"]) == set(animals.loc[animals["weight"] > 100, "name"])
print("PASS tính chất kết quả: > 100, giảm dần, đủ tên")

# Đầu vào không bị sửa
before = animals.copy()
for key, fn in impls.items():
    fn(animals)
findHeavyAnimals(animals)
pd.testing.assert_frame_equal(animals, before)
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: >= thay vì >
wrong = animals.assign(weight=[464, 41, 100, 463, 50, 349])
assert wrong[wrong["weight"] >= 100].shape[0] == 4 and wrong[wrong["weight"] > 100].shape[0] == 3
print("PASS phản ví dụ: có con đúng 100 kg thì >= cho kết quả khác >")

# Phản ví dụ 2: không đặt ascending=False -> thứ tự tăng dần
asc = animals[animals["weight"] > 100].sort_values("weight")[["name"]]
assert asc["name"].tolist() == ["Alex", "Tommy", "Jonathan", "Tatiana"]
print("PASS phản ví dụ: thiếu ascending=False cho", asc["name"].tolist())

PASS bảng rỗng -> DataFrame rỗng có cột name
PASS kiểu trả về là DataFrame, cột: ['name']
PASS tính chất kết quả: > 100, giảm dần, đủ tên
PASS đầu vào không bị sửa
PASS phản ví dụ: có con đúng 100 kg thì >= cho kết quả khác >
PASS phản ví dụ: thiếu ascending=False cho ['Alex', 'Tommy', 'Jonathan', 'Tatiana']


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Dữ liệu có nhiều `weight` trùng (khoảng 0 đến 300) để thử cả tình huống bằng nhau; các cách có `kind="stable"` phải khớp **cả thứ tự**.

In [24]:
import random

random.seed(2891)
for trial in range(300):
    n = random.randint(0, 15)
    rows = [(f"n{i}", random.choice([0, 50, 99, 100, 101, 150, 200, 300, random.randint(0, 400)])) for i in range(n)]
    df = make(rows)
    ref = df[df["weight"] > 100].sort_values("weight", ascending=False, kind="stable")[["name"]]
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df), ref)
    # bản nộp không cần ổn định: chỉ so sánh tập (name, weight) theo thứ tự weight giảm dần
    sub = findHeavyAnimals(df)
    assert sub["name"].tolist() == [] or df.loc[sub.index, "weight"].is_monotonic_decreasing
    assert set(sub["name"]) == set(ref["name"])
print("PASS 300 bảng ngẫu nhiên - mọi cách khớp nhau; bản nộp đúng tập tên và thứ tự weight")

PASS 300 bảng ngẫu nhiên - mọi cách khớp nhau; bản nộp đúng tập tên và thứ tự weight


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chọn cột `name` quá sớm

In [25]:
try:
    sai = animals[["name"]][animals["weight"] > 100].sort_values("weight", ascending=False)
    print(sai)
except KeyError as e:
    print("KeyError:", e)
dung = animals[animals["weight"] > 100].sort_values("weight", ascending=False)[["name"]]
print("đúng:", dung["name"].tolist())

KeyError: 'weight'
đúng: ['Tatiana', 'Jonathan', 'Tommy', 'Alex']


**Vì sao sai:** nếu chọn `[["name"]]` trước, bảng chỉ còn cột `name`, **không còn `weight`** để sắp xếp (`KeyError`). (Ở dòng đầu, mặt nạ vẫn tính được vì lấy từ `animals` gốc, nhưng `sort_values("weight")` thì không.) Thứ tự đúng: lọc và sắp xếp trước, **chọn cột sau cùng**.

### Lỗi 2: Dùng `>=` thay vì `>`

In [26]:
sai = animals[animals["weight"] >= 100]
print("không khác với dữ liệu này, nhưng thử thêm con đúng 100 kg:")
more = pd.concat([animals, pd.DataFrame({"name": ["Zed"], "species": ["Ox"], "age": [3], "weight": [100]})], ignore_index=True)
print("sai  (>=):", more[more["weight"] >= 100]["name"].tolist())
print("đúng (> ):", more[more["weight"] > 100]["name"].tolist())

không khác với dữ liệu này, nhưng thử thêm con đúng 100 kg:
sai  (>=): ['Tatiana', 'Alex', 'Jonathan', 'Tommy', 'Zed']
đúng (> ): ['Tatiana', 'Alex', 'Jonathan', 'Tommy']


**Vì sao sai:** đề nói **"nặng hơn 100 kg"**, tức **lớn hơn hẳn** 100. Con vật đúng 100 kg không thỏa. Luôn đọc kỹ từ khóa: "hơn" là `>`, "ít nhất" là `>=`.

### Lỗi 3: Quên `ascending=False`

In [27]:
sai = animals[animals["weight"] > 100].sort_values("weight")[["name"]]
print("sai  :", sai["name"].tolist())       # tăng dần: Alex trước
dung = animals[animals["weight"] > 100].sort_values("weight", ascending=False)[["name"]]
print("đúng :", dung["name"].tolist())

sai  : ['Alex', 'Tommy', 'Jonathan', 'Tatiana']
đúng : ['Tatiana', 'Jonathan', 'Tommy', 'Alex']


**Vì sao sai:** `sort_values` mặc định **tăng dần**. Đề yêu cầu **giảm dần** (nặng nhất trước), nên cần `ascending=False`.

### Lỗi 4: Trả về Series thay vì DataFrame

In [28]:
sai = animals[animals["weight"] > 100].sort_values("weight", ascending=False)["name"]
print("sai  : kiểu", type(sai).__name__)
dung = animals[animals["weight"] > 100].sort_values("weight", ascending=False)[["name"]]
print("đúng : kiểu", type(dung).__name__)

sai  : kiểu Series
đúng : kiểu DataFrame


**Vì sao sai:** `["name"]` trả về **Series**; chữ ký yêu cầu `DataFrame`, nên dùng `[["name"]]` (danh sách bên trong).

### Lỗi 5: `inplace=True` làm đứt chuỗi

In [29]:
tmp = animals.copy()
try:
    tmp.sort_values("weight", inplace=True)[["name"]]
except TypeError as e:
    print("TypeError:", e)
r = tmp.sort_values("weight", inplace=True)
print("kết quả của lệnh inplace:", r)           # None

TypeError: 'NoneType' object is not subscriptable
kết quả của lệnh inplace: None


**Vì sao sai:** `inplace=True` **sửa tại chỗ** và **trả `None`**; viết tiếp `.something` sau đó là gọi phương thức trên `None` (hoặc truy cập `None[...]`), chuỗi bị đứt. Method chaining chỉ hoạt động khi **mỗi bước trả về một DataFrame**; vì vậy tránh `inplace=True`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "lọc, sắp xếp, lấy vài cột" → nghĩ tới chuỗi **lọc -> `sort_values` -> chọn cột**.
- Công thức nhớ: **`(df[điều_kiện].sort_values("cột", ascending=False)[["cột_cần"]])`**.
- Viết chuỗi dài trong **dấu ngoặc đơn** để xuống dòng, mỗi dòng một bước, có thể thêm chú thích riêng từng dòng.
- Thứ tự chuẩn của các bước: **lọc hàng sớm** (ít dữ liệu hơn cho các bước sau) -> **sắp xếp** -> **chọn cột**.
- Mỗi bước phải **trả về DataFrame**; tránh `inplace=True`.
- Cần `top k`: `nlargest(k, "weight")`; cần chèn bước tự viết: `.pipe(hàm)`; cần tạo cột mới giữa chuỗi: `.assign(...)`.
- Có thể thêm `.reset_index(drop=True)` ở cuối nếu đề muốn nhãn `0..n-1`.

## 15. Biến thể của bài

1. Chỉ lấy **3 con nặng nhất**: `... .sort_values("weight", ascending=False).head(3)` hoặc `df.nlargest(3, "weight")`.
2. Lấy cả `name` và `weight`: `[["name", "weight"]]`.
3. Nặng hơn 100 **và** tuổi dưới 50: `df.query("weight > 100 and age < 50")`.
4. Sắp xếp theo **hai tiêu chí**: `sort_values(["weight", "name"], ascending=[False, True])`.
5. Đánh lại nhãn `0..n-1`: `.reset_index(drop=True)`.
6. Chuyển thành **danh sách tên** (không phải DataFrame): `... ["name"].tolist()`.

Ta thử vài biến thể:

In [30]:
print("1. top 3 :", animals.nlargest(3, "weight")["name"].tolist())
print("3. 2 điều kiện:", animals.query("weight > 100 and age < 50")["name"].tolist())
print("4. 2 tiêu chí :", animals.sort_values(["weight", "name"], ascending=[False, True])["name"].tolist()[:3])
chain = (
    animals.query("weight > 100")
    .sort_values("weight", ascending=False)
    [["name", "weight"]]
    .reset_index(drop=True)
)
print("2+5. name+weight, nhãn mới:")
print(chain)
print("6. danh sách tên:", findHeavyAnimals(animals)["name"].tolist())

1. top 3 : ['Tatiana', 'Jonathan', 'Tommy']
3. 2 điều kiện: ['Alex', 'Jonathan', 'Tommy']
4. 2 tiêu chí : ['Tatiana', 'Jonathan', 'Tommy']
2+5. name+weight, nhãn mới:
       name  weight
0   Tatiana     464
1  Jonathan     463
2     Tommy     349
3      Alex     328
6. danh sách tên: ['Tatiana', 'Jonathan', 'Tommy', 'Alex']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + `sorted` | tự lọc, tự sắp xếp | `O(m + h log h)` (Python, chậm) | `O(h)` | hiểu bản chất |
| 2. `df[mask].sort_values()[[cột]]` | chuỗi mặt nạ | `O(m + h log h)` | `O(m + h)` | **cách nộp bài** |
| 3. `df.query(...).sort_values()[[cột]]` | lọc bằng chuỗi biểu thức | `O(m + h log h)` | `O(m + h)` | điều kiện phức tạp, đọc gọn |
| 4. `loc[lambda d: ...]` ... `loc[:, [cột]]` | lọc/chọn cột bằng `loc` | `O(m + h log h)` | `O(m + h)` | chuỗi không cần đặt tên biến trung gian |
| 5. `sort_values` -> `pipe(lọc)` -> `filter(items=)` | sắp xếp trước, chèn hàm | `O(m log m)` | `O(m)` | cần chèn hàm tùy biến; cho thấy đổi thứ tự bước |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(animals["weight"] > 100).sum()`, `animals.sort_values("weight")["name"].iloc[0]`, `type(animals["name"]).__name__`, `type(animals[["name"]]).__name__`, `animals.query("weight > 300 and age < 50")["name"].tolist()`.

**Bài 2 (tính tay).** Với `weight = [10, 300, 100, 150, 300]` và `name = [a, b, c, d, e]`: viết mặt nạ `> 100`, nhãn các hàng còn lại, thứ tự tên sau khi sắp xếp giảm dần ổn định.

**Bài 3 (điền TODO).** Hoàn thành `heavy_todo(df)` bằng chuỗi.

In [31]:
def heavy_todo(df):
    # TODO: return (
    #     df[df["weight"] > 100]
    #     .sort_values(...)     # giảm dần theo weight
    #     [[...]]               # chỉ cột name
    # )
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `heavy_loop2(animals)` bằng vòng lặp thuần Python (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `top_heavy(animals, k)` trả về **`k` con nặng nhất** (vẫn phải nặng hơn 100), cột `name` và `weight`, nhãn `0..k-1`. (b) Viết `species_total_weight(animals)` trả về DataFrame `species`, `total_weight` chỉ cho các loài **có tổng cân nặng lớn hơn 300**, sắp giảm dần theo `total_weight`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `4`, `'Khaled'`, `'Series'`, `'DataFrame'`, và với `query("weight > 300 and age < 50")` hãy kiểm từng con: Alex (328, tuổi 6), Jonathan (463, tuổi 45), Tommy (349, tuổi 26) thỏa; Tatiana (tuổi 98) không. Bài 2: so sánh `> 100` cho từng phần tử.
- **Mức 2 (vừa):** Bài 2: mặt nạ `[F, T, F, T, T]`, nhãn `[1, 3, 4]`, thứ tự giảm dần ổn định `b, e, d`. Bài 3: `ascending=False`, `[["name"]]`. Bài 4: tạo list `(weight, nhãn)` cho hàng thỏa, sắp xếp giảm dần, `animals.loc[nhãn, ["name"]]`.
- **Mức 3 (sâu):** Bài 5(a): `df[df["weight"] > 100].sort_values("weight", ascending=False).head(k)[["name", "weight"]].reset_index(drop=True)`. Bài 5(b): `df.groupby("species", as_index=False)["weight"].sum().rename(columns={"weight": "total_weight"})` rồi lọc `> 300` và `sort_values`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert (animals["weight"] > 100).sum() == 4
assert animals.sort_values("weight")["name"].iloc[0] == "Khaled"
assert type(animals["name"]).__name__ == "Series" and type(animals[["name"]]).__name__ == "DataFrame"
assert animals.query("weight > 300 and age < 50")["name"].tolist() == ["Alex", "Jonathan", "Tommy"]

# Bài 2
b2 = pd.DataFrame({"name": list("abcde"), "weight": [10, 300, 100, 150, 300]})
assert (b2["weight"] > 100).tolist() == [False, True, False, True, True]
r = b2[b2["weight"] > 100]
assert r.index.tolist() == [1, 3, 4]
assert r.sort_values("weight", ascending=False, kind="stable")["name"].tolist() == ["b", "e", "d"]

# Bài 3
def heavy_todo_ans(df):
    return (
        df[df["weight"] > 100]
        .sort_values("weight", ascending=False)
        [["name"]]
    )

pd.testing.assert_frame_equal(heavy_todo_ans(animals), expected_example)

# Bài 4
def heavy_loop2(animals):
    heavy = []
    for label, weight in zip(animals.index, animals["weight"]):
        if weight > 100:
            heavy.append((weight, label))
    heavy.sort(key=lambda pair: -pair[0])
    return animals.loc[[label for _, label in heavy], ["name"]]

pd.testing.assert_frame_equal(heavy_loop2(animals), expected_example)

# Bài 5(a)
def top_heavy(animals, k):
    return (
        animals[animals["weight"] > 100]
        .sort_values("weight", ascending=False)
        .head(k)[["name", "weight"]]
        .reset_index(drop=True)
    )

t = top_heavy(animals, 2)
assert t["name"].tolist() == ["Tatiana", "Jonathan"] and t["weight"].tolist() == [464, 463] and t.index.tolist() == [0, 1]

# Bài 5(b)
def species_total_weight(animals):
    g = animals.groupby("species", as_index=False)["weight"].sum().rename(columns={"weight": "total_weight"})
    return g[g["total_weight"] > 300].sort_values("total_weight", ascending=False).reset_index(drop=True)

sp = species_total_weight(animals)
assert sp["species"].tolist() == ["Snake", "Monkey", "Panda", "Leopard"]
assert sp["total_weight"].tolist() == [464, 463, 349, 328]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lọc -> sắp xếp -> chọn cột bằng method chaining)

```python
import pandas as pd

def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame:
    return (
        animals[animals["weight"] > 100]
        .sort_values("weight", ascending=False)
        [["name"]]
    )
```

**Checklist**

- [ ] Điều kiện là `weight > 100` (lớn hơn **hẳn**, không phải `>=`)?
- [ ] `sort_values("weight", ascending=False)` (giảm dần)?
- [ ] Chọn cột **sau cùng** bằng `[["name"]]` (hai cặp ngoặc, trả DataFrame, không phải Series)?
- [ ] Mỗi mắt xích trả về DataFrame (không dùng `inplace=True` giữa chuỗi)?
- [ ] Biết nhãn gốc được giữ; chỉ `reset_index(drop=True)` nếu đề yêu cầu?
- [ ] Đã thử: không con nào > 100, đúng 100 kg, `weight` trùng nhau, bảng rỗng?
- [ ] Skeleton đúng `def findHeavyAnimals(animals: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?